In [1]:
import os, json, time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

print("환경 설정 완료")

환경 설정 완료


In [ ]:
# 서울 날씨 알려줘? -> LLM -> tool_calls -> 개발자가 만든 get_weather("서울") -> result - > 서울은 맑아요.
# api calls

In [2]:
def get_weather(city):
    weather_db = {
        "seoul" : "맑음, 22도, 습도 45%",
        "tokyo" : "흐림, 19도, 습도 70%",
        "new york" : "비, 15도, 습도 80%"
    }
    return weather_db.get(city, f"{city}: 정보 없음")

In [3]:
tool_calls = {"name" : "get_weather", "args" :{"city" : "seoul"}}
result = get_weather(**tool_calls['args'])

In [4]:
result

'맑음, 22도, 습도 45%'

In [20]:
@tool
def get_weather(city  : str) -> str:
    """날씨를 조회하는 함수입니다"""
    weather_db = {
        "seoul" : "맑음, 22도, 습도 45%",
        "도쿄" : "흐림, 19도, 습도 70%",
        "뉴욕" : "비, 15도, 습도 80%"
    }
    return weather_db.get(city, f"{city}: 정보 없음")


In [29]:
@tool
def get_exchange_rate(from_currency : str, to_currency: str) -> str:
    """두 통화 간의 환율을 조회합니다"""
    currency_db = {
        ("KRW", "USD") : 0.00080, ("KRW", "JPY") : 0.11,
        ("USD", "KRW") : 1400, ("JPY", "KRW") : 9.0
    }
    return f"1 {from_currency} = {currency_db.get((from_currency, to_currency))} {to_currency}"

llm_with_tool = llm.bind_tools([get_exchange_rate])
llm_with_tool.invoke("원달러 환율 알려줘")


AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 62, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e3fec59ca6', 'id': 'chatcmpl-DX3cEq0QcRNOFe7iN6bEClinAjf4d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafcf-4feb-7d72-99fd-8df517c6e1a2-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'USD'}, 'id': 'call_fmT6PBOPfctky8XtC8jZJBlz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 22, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0

In [31]:
response = llm_with_tool.invoke("원달러 환율 알려줘")
response.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'from_currency': 'KRW', 'to_currency': 'USD'},
  'id': 'call_Ki5TXv2UCAvHGBxRqRcLnzFU',
  'type': 'tool_call'}]

In [34]:
type(get_exchange_rate)

langchain_core.tools.structured.StructuredTool

In [36]:
type(get_exchange_rate.name)

str

In [37]:
tools = [get_exchange_rate]
tool_map = {t.name:t for t in tools}

if response.tool_calls:
    tc = response.tool_calls[0]
    tool = tool_map.get(tc['name'])
    result = tool.invoke(tc['args'])

result

'1 KRW = 0.0008 USD'

In [21]:
get_weather.name

'get_weather'

In [22]:
get_weather.description

'날씨를 조회하는 함수입니다'

In [23]:
get_weather.args

{'city': {'title': 'City', 'type': 'string'}}

In [24]:
llm_with_tools = llm.bind_tools([get_weather])

In [25]:
result = llm_with_tools.invoke("서울 날씨 어때?")

In [ ]:
result

In [26]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'city': '서울'},
  'id': 'call_oiuTE8BQQlpotglTIYaYeFzp',
  'type': 'tool_call'}]

In [27]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_124771d82f', 'id': 'chatcmpl-DX3OmConfuv3TuUOBpdyZTQ5Ci61a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafc2-997b-7df3-a019-6bf86f4b3504-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_oiuTE8BQQlpotglTIYaYeFzp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [28]:
if result.tool_calls:
    tc = result.tool_calls[0]
    print(f"tool : {tc['name']}")
    
    result_final = get_weather.invoke(tc['args'])
    print(f"result : {result_final}")
    

tool : get_weather
result : 맑음, 22도, 습도 45%


In [38]:
type(get_exchange_rate)

langchain_core.tools.structured.StructuredTool

In [ ]:
# StructuredTool.from_function

In [39]:
def search_etf_func(category, min_return=0, max_expense=1.0):
    etfs = [
        {"name": "KODEX 200", "cat": "국내주식", "ret": 8.5, "exp": 0.15},
        {"name": "KODEX S&P500TR", "cat": "해외주식", "ret": 25.3, "exp": 0.05},
        {"name": "TIGER 나스닥100", "cat": "해외주식", "ret": 30.2, "exp": 0.07},
        {"name": "ACE 미국배당다우존스", "cat": "배당", "ret": 12.1, "exp": 0.01},
    ]
    
    result = [e for e in etfs if e['cat'] == category and e['ret'] >= min_return and e['exp'] <=max_expense]
    return json.dumps(result, ensure_ascii =False) if result else '조건에 맞는 ETF 없음'

In [50]:
from pydantic import BaseModel, Field

class ETFSearchInput(BaseModel):
    category: str= Field(description="ETF 카테고리")
    min_return: float = Field(default=0, ge=0, le=100, description='최소 수익률')
    max_expense: float = Field(default=1.0, ge=0, le=5, description='최대 운용보수')

In [51]:
search_etf = StructuredTool.from_function(
                func = search_etf_func, name='search_etf',
                description= '조건에 맞는 상품을 검색합니다.',
                args_schema = ETFSearchInput,
            )

search_etf

StructuredTool(name='search_etf', description='조건에 맞는 상품을 검색합니다.', args_schema=<class '__main__.ETFSearchInput'>, func=<function search_etf_func at 0x7feff63fd1c0>)

In [52]:
llm_etf = llm.bind_tools([search_etf])
queries = ["해외주식 ETF 추천해줘"]
for q in queries:
    resp = llm_etf.invoke(q)
    if resp.tool_calls:
        tc = resp.tool_calls[0]
        result = search_etf.invoke(tc['args'])
        print(f"Q : {q}")
        print(tc['args'])
        print(f"A : {result}")
    

Q : 해외주식 ETF 추천해줘
{'category': '해외주식'}
A : [{"name": "KODEX S&P500TR", "cat": "해외주식", "ret": 25.3, "exp": 0.05}, {"name": "TIGER 나스닥100", "cat": "해외주식", "ret": 30.2, "exp": 0.07}]


In [58]:
class EmployeeSearchInput(BaseModel):
    department : str = Field(description="부서명", enum=["개발", "마케팅", "영업"])
    min_salary : int = Field(default=3000, ge=3000, description="최소 연봉(만원)")

def search_employee(department, min_salary=3000):
    employees = [
        {"name" : "김철수" , "dept" : "개발", "salary" : 5000},
        {"name" : "이영희" , "dept" : "마케팅", "salary" : 3000},
        {"name" : "박수철" , "dept" : "영업", "salary" : 4000},
        {"name" : "신희영" , "dept" : "개발", "salary" : 6000}
    ]
    result = [e for e in employees if e['dept'] == department and e['salary'] >= min_salary ]
    return json.dumps(result, ensure_ascii =False) if result else '조건에 맞는 직원 없음'

emp_tool = StructuredTool.from_function(
            func = search_employee, name = 'search_employee',
            description ='부서와 연봉 조건으로 직원 검색', args_schema = EmployeeSearchInput
            )
llm_emp = llm.bind_tools([emp_tool])
resp = llm_emp.invoke("연봉 5000 이상 개발자 보여주세요")
if resp.tool_calls:
    result = emp_tool.invoke(resp.tool_calls[0]['args'])
    print(resp.tool_calls[0])
    print(result)

/tmp/ipykernel_1428472/4159724441.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  department : str = Field(description="부서명", enum=["개발", "마케팅", "영업"])


{'name': 'search_employee', 'args': {'department': '개발', 'min_salary': 5000}, 'id': 'call_Bt07BC0rk5qYwSF1RxdevmNX', 'type': 'tool_call'}
[{"name": "김철수", "dept": "개발", "salary": 5000}, {"name": "신희영", "dept": "개발", "salary": 6000}]


In [44]:
llm_etf = llm.bind_tools([search_etf])
resp = llm_etf.invoke("해외주식 ETF 추천해줘")

In [45]:
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 125, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_998d5473a0', 'id': 'chatcmpl-DX3s1i455eeRKDipipF3sZVtlDGXs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafde-4301-7ff3-9542-466605b976b8-0', tool_calls=[{'name': 'search_etf', 'args': {'category': '해외주식'}, 'id': 'call_y6gzsfFXPDX0AVmtXDFK8n8N', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 18, 'total_tokens': 143, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [47]:
llm_etf = llm.bind_tools([search_etf])
queries = ["해외주식 ETF 추천해줘", "수수료 낮은 배당 ETF 찾아줘"]
for q in queries:
    resp = llm_etf.invoke(q)
    if resp.tool_calls:
        tc = resp.tool_calls[0]
        result = search_etf.invoke(tc['args'])
        print(f"Q : {q}")
        print(tc['args'])
        print(f"A : {result}")
    

Q : 해외주식 ETF 추천해줘
{'category': '해외주식'}
A : 조건에 맞는 ETF 없음
Q : 수수료 낮은 배당 ETF 찾아줘
{'category': '배당', 'max_expense': 0.5}
A : [{"name": "ACE 미국배당다우존스", "cat": "배당", "ret": 12.1, "exp": 0.01}]


In [ ]:
# get_weather_tool : 날씨
# get_exchange_rate : 환율
    
# 서울과 도쿄 날씨 비교하고 환율도 알려줘

# tool_calls : [
#     {"name" : 'get_weather_tool', 'args' : {city : seoul}},
#     {"name" : 'get_weather_tool', 'args' : {city : japan}},
#     {"name" : 'get_exchange_rate', 'args' :},
    
# ]

In [60]:
all_tools = [get_weather, get_exchange_rate, search_etf]
llm_multi = llm.bind_tools(all_tools)

In [61]:
resp = llm_multi.invoke("서울과 도쿄 날씨 비교하고 환율도 알려줘")
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 187, 'total_tokens': 255, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'id': 'chatcmpl-DX4HRRWYNwXPjSJPQQ1jQPewWhJxw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daff6-50fa-72b1-a526-27f49960b199-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_ZoPV2kdl5W2ejN6fRstUDNn8', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_QBq9WyRx2Pv8sbu8FFkgElWX', 'type': 'tool_call'}, {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_AoF2WhODBQMBSiP60S9

In [63]:
tool_map = {t.name:t for t in all_tools}
for tc in resp.tool_calls:
    result = tool_map[tc['name']].invoke(tc['args'])
    print(f" {tc['name']} ({tc['args']}) -> {result}")

 get_weather ({'city': '서울'}) -> 맑음, 22도, 습도 45%
 get_weather ({'city': '도쿄'}) -> 흐림, 19도, 습도 70%
 get_exchange_rate ({'from_currency': 'KRW', 'to_currency': 'JPY'}) -> 1 KRW = 0.11 JPY


In [ ]:
# 1. 계산기 툴을 추가
# 2. get_weather, get_exchange_rate, 계산기
# 3. 서울 날씨 알려주고 15*24 계산해줘

In [65]:
from langchain_core.tools import tool, StructuredTool

@tool
def calculate(expression):
    """수학 계산식을 평가합니다"""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except:
        return "error"
    
tools = [get_weather, get_exchange_rate, calculate]
llm_3 = llm.bind_tools(tools)
tool_map_3 = {t.name:t for t in tools}
resp = llm_3.invoke("서울 날씨 알려주고 15*24 계산해줘")
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 110, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DX4VuCMV5svjMndTFepXH15xnap2N', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db004-02ab-71d3-838d-1d5ddcbee07b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_MupyJrU2JmHZHZmlPwhGStLD', 'type': 'tool_call'}, {'name': 'calculate', 'args': {'expression': '15*24'}, 'id': 'call_TfXCH9LABY8ChdHQayDyLjUn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 110, 'output_tokens': 45, 'total_tokens': 155, 'input_

In [68]:
all_tools = [get_weather, get_exchange_rate]
llm_with_tools = llm.bind_tools(all_tools)

query = "서울 날씨 알려주고, 원-엔 환율도 알려줘"
messages = [HumanMessage(content=query)]
ai_msg = llm_with_tools.invoke(messages)

messages.append(ai_msg)

tool_map = {t.name:t for t in all_tools}

for tc in ai_msg.tool_calls:
    selected_tool = tool_map[tc['name']]
    tool_result = selected_tool.invoke(tc['args'])
    
    messages.append(ToolMessage(content = str(tool_result), tool_call_id = tc['id']))

messages

[HumanMessage(content='서울 날씨 알려주고, 원-엔 환율도 알려줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 92, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_83e2dd34fc', 'id': 'chatcmpl-DX4mqbBHG96PQwXPknJF5STAkKSHx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db014-08eb-70e0-88ca-1f3cee00e90a-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_iG3ueI3krOtfrlnhOhSB7Khv', 'type': 'tool_call'}, {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_u6YTvECARqKZLZ6Ip6BOFyPP', 'type':

In [69]:
final_answer = llm_with_tools.invoke(messages)
final_answer

AIMessage(content='현재 서울의 날씨는 맑고, 기온은 22도, 습도는 45%입니다.\n\n또한, 원-엔 환율은 1 KRW = 0.11 JPY입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 181, 'total_tokens': 230, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_83e2dd34fc', 'id': 'chatcmpl-DX4nFREHpmpxgnUt6s7BRsvwMKvZu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db014-6af9-7513-ae4d-5c9a4a556b7c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 49, 'total_tokens': 230, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [70]:
final_answer.content

'현재 서울의 날씨는 맑고, 기온은 22도, 습도는 45%입니다.\n\n또한, 원-엔 환율은 1 KRW = 0.11 JPY입니다.'

In [71]:
def tool_loop(query, tools, max_turns=5): # trade-off, 예외처리/방어로직
    tool_map = {t.name:t for t in tools}
    llm_t = llm.bind_tools(tools)
    messages = [HumanMessage(content=query)]
    
    for turn in range(max_turns):
        response = llm_t.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            return {"answer" : response.content, "log" : call_log}
        
        for tc in response.tool_calls:
            tool_obj = tool_map[tc['name']]
            result = tool_obj.invoke(tc['args']) if tool_obj else "도구 없음"
            messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
            print(f" [Turn {turn+1}] {tc['name']}, {tc['args']} -> {result}")
            
    return {"answer": "최대 턴 초과", "log" : call_log}

In [72]:
answer = tool_loop('서울 날씨 알려주고, 원-엔 환율도 알려줘', [get_weather, get_exchange_rate])
answer

 [Turn 1] get_weather, {'city': '서울'} -> 맑음, 22도, 습도 45%
 [Turn 1] get_exchange_rate, {'from_currency': 'KRW', 'to_currency': 'JPY'} -> 1 KRW = 0.11 JPY


'서울의 날씨는 맑고, 기온은 22도, 습도는 45%입니다. \n\n또한, 현재 원-엔 환율은 1 KRW = 0.11 JPY입니다.'

In [73]:
def tool_loop_with_log(query, tools, max_turns=5): # trade-off, 예외처리/방어로직
    tool_map = {t.name:t for t in tools}
    llm_t = llm.bind_tools(tools)
    messages = [HumanMessage(content=query)]
    call_log = []
    
    for turn in range(max_turns):
        response = llm_t.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            return {"answer" : response.content, "log" : call_log}
        
        for tc in response.tool_calls:
            tool_obj = tool_map[tc['name']]
            result = tool_obj.invoke(tc['args']) if tool_obj else "도구 없음"
            call_log.append({
                "turn" : turn+1, "tool" : tc['name'], 'args' : tc['args'], 'result' : result
            })
            messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
#             print(f" [Turn {turn+1}] {tc['name']}, {tc['args']} -> {result}")
            
    return {"answer": "최대 턴 초과", "log" : call_log}

In [74]:
answer = tool_loop_with_log('서울 날씨 알려주고, 원-엔 환율도 알려줘', [get_weather, get_exchange_rate])
answer

{'answer': '현재 서울의 날씨는 맑으며, 기온은 22도이고 습도는 45%입니다. \n\n또한, 원-엔 환율은 1 KRW가 0.11 JPY입니다.',
 'log': [{'turn': 1,
   'tool': 'get_weather',
   'args': {'city': '서울'},
   'result': '맑음, 22도, 습도 45%'},
  {'turn': 1,
   'tool': 'get_exchange_rate',
   'args': {'from_currency': 'KRW', 'to_currency': 'JPY'},
   'result': '1 KRW = 0.11 JPY'}]}

In [ ]:
# 현재 서울의 날씨는 맑으며, 기온은 220도이고  

In [75]:
class WeatherReport(BaseModel):
    city: str = Field(description='city name')
    temperature: int = Field(description = 'temperature')
    condition: str = Field(descriptioin = 'weather cond')
    recommendation : str = Field(description = 'activity recommend')

structured_llm = llm.with_structured_output(WeatherReport)
report = structured_llm.invoke("서울의 봄 날씨를 보고서로 작성해줘")
report

/tmp/ipykernel_1428472/586593510.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptioin'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  condition: str = Field(descriptioin = 'weather cond')
/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=WeatherReport(city='서...닉을 즐기세요.'), input_type=WeatherReport])
  return self.__pydantic_serializer__.to_python(


WeatherReport(city='서울', temperature=17, condition='맑음', recommendation='야외에서 산책이나 피크닉을 즐기세요.')

In [76]:
report.city, report.temperature

('서울', 17)

In [78]:
llm_forced = llm.bind_tools([get_weather, calculate], tool_choice='get_weather')
resp = llm_forced.invoke("안녕하세요")
resp.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Seoul'},
  'id': 'call_NlVyrqlaCJlpK4fxZEjdo4Kh',
  'type': 'tool_call'}]

In [79]:
structured_llm = llm.with_structured_output(WeatherReport)

In [80]:
query = "안녕? 반가워"
messages = [HumanMessage(content=query)]

resp = llm_forced.invoke(messages)
messages.append(resp)

if resp.tool_calls:
    tc = resp.tool_calls[0]
    tool_result = get_weather.invoke(tc['args'])
    
    messages.append(ToolMessage(content = str(tool_result), tool_call_id = tc['id']))
    
final_report = structured_llm.invoke(messages)
final_report

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=WeatherReport(city='Seoul... 좋은 날입니다.'), input_type=WeatherReport])
  return self.__pydantic_serializer__.to_python(


WeatherReport(city='Seoul', temperature=20, condition='맑음', recommendation='공원에서 산책하기 좋은 날입니다.')

In [83]:
class ETFAnalysis(BaseModel):
    etf_name :str = Field(description='ETF 상품명')
    risk_level :str = Field(description='위험 등급')
    expected_return: float = Field(description='예상 연간 수익률')
    recommendation: str = Field(description='투자 추천 의견')
        
etf_llm = llm.with_structured_output(ETFAnalysis)
analysis = etf_llm.invoke("KODEX S&P500TR은 미국 S&P500 지수를 추종하는 ETF입니다 운용보수는 0.05%, 최근 수익률은 25.3%입니다. 분석해주세요")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ETFAnalysis(etf_name='KOD...기 투자로 적합.'), input_type=ETFAnalysis])
  return self.__pydantic_serializer__.to_python(


In [84]:
analysis

ETFAnalysis(etf_name='KODEX S&P500TR', risk_level='중간', expected_return=25.3, recommendation='미국 주식 시장의 성장을 고려할 때, 장기 투자로 적합.')